# Hindcast 0008-01: O3 And U60N10 Evolution

拆出 poster-style O3 column evolution 和 60N 10 hPa 风速演变图，包括 JAN INITIAL ONLY 的 all/best5/worst5 mean。

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EP_WINDOW_LABEL = "Jan20-Feb10"
AO_BASE_WINDOW = EP_WINDOW
AO_BASE_WINDOW_LABEL = EP_WINDOW_LABEL
AO_TEST_WINDOWS = {
    "Jan20-Feb10": EP_WINDOW,
    "Jan": ((1, 1), (1, 31)),
    "Jan-Feb": ((1, 1), (2, 28)),
    "Jan-Mar": ((1, 1), (3, 31)),
}
Z300_WINDOW = EP_WINDOW
Z300_WINDOW_LABEL = EP_WINDOW_LABEL
MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
    "Apr": ((4, 1), (4, 30)),
    "May": ((5, 1), (5, 30)),
}
MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May"]

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_EP_HPA = PLEV_EP_PA / 100.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0
PLEV_Z300_HPA = PLEV_Z300_PA / 100.0

EP_SCALAR_DESCRIPTION = (
    f"mean -ep2 vertical EP-flux component at {PLEV_EP_HPA:.0f} hPa, "
    f"cos-lat mean {LAT_EP[0]:.0f}-{LAT_EP[1]:.0f}N, {EP_WINDOW_LABEL}; not EP-flux divergence"
)
Z300_PATTERN_DESCRIPTION = (
    f"{PLEV_Z300_HPA:.0f} hPa height, {Z300_WINDOW_LABEL} mean, "
    f"{LAT_Z300[0]:.0f}-{LAT_Z300[1]:.0f}N, zonal mean removed before pattern metrics"
)

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
    "wave1_plus_wave2": "Wave 1 + Wave 2",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_B2000_YEARS_FOR_Z300 = None  # None = all B2000WCN001002 Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

In [ ]:
# -----------------------------
# Figure routing for the split notebook
# -----------------------------
NOTEBOOK_STEM = "Hindcast_0008_01_04_o3_u_evolution"

def use_figure_dir(content_tag: str):
    global FIG_DIR
    FIG_DIR = OUT_ROOT / "figures" / NOTEBOOK_STEM / content_tag
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print("Figure dir:", FIG_DIR)
    return FIG_DIR


In [ ]:
# -----------------------------
# Shared scalar data prep for split diagnostics
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)
ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all["EP100_wave1_plus_wave2"] = rmse_ep_all["EP100_wave1"] + rmse_ep_all["EP100_wave2"]
EP_SCALAR_COLS = [
    ("EP100_all_waves", "all_waves", "All", "black"),
    ("EP100_wave1", "wave1", "W1", "tab:blue"),
    ("EP100_wave2", "wave2", "W2", "tab:orange"),
    ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
]
print("Prepared base tables:", rmse_ep_all.shape)


## 图：Hindcast O3/U 演变与 JAN INITIAL ONLY best/worst

**做什么**：复刻 poster-style 的 Hindcast O3 column evolution，并加入 60N 10 hPa zonal wind；JAN INITIAL ONLY 图中同时画 all-30 mean、best5 mean、worst5 mean。

**怎么做**：best/worst 按 Jan1-May30 的 O3 RMSE 排序。O3 使用 cleaned partial O3；U60N10 从 raw U 插值到 10 hPa 并做 zonal mean。

**科学问题**：可预报性好的/差的成员在臭氧和极夜急流演变上何时分离？

**预期**：worst5 可能更早出现波驱动相关的风场/臭氧偏离。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("01_o3_u60n10_evolution")


In [ ]:
# -----------------------------
# Poster-style evolution plots from cleaned products.
# O3 uses new partial_O3. U60N10 uses raw Hindcast U and a local cache.
# This block also draws the JAN INITIAL ONLY all-30, best-5, and worst-5 means on the same plot.
# Best/worst are ranked by full-window O3 RMSE from the cleaned partial_O3 product.
# -----------------------------

ranked_members = rmse_ep_all.sort_values("RMSE_DU")["member"].tolist()
best5_members = ranked_members[:5]
worst5_members = ranked_members[-5:]
print("JAN INITIAL ONLY best5 by O3 RMSE:", best5_members)
print("JAN INITIAL ONLY worst5 by O3 RMSE:", worst5_members)


def subset_members_da(da: xr.DataArray, members: Sequence[str]) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    member_vals = [str(v) for v in da["member"].values]
    keep_idx = []
    requested = set(members)
    for i, v in enumerate(member_vals):
        short = member_short_id(v)
        if short in requested or v in requested:
            keep_idx.append(i)
    if not keep_idx:
        raise ValueError(f"No requested members found. requested={members}, available sample={member_vals[:5]}")
    return da.isel(member=keep_idx)


def load_case_o3_for_evolution(case):
    da, date = load_hindcast_o3(case)
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_ref_o3_curve(year=REF_YEAR):
    da, date = load_bwcn_ref_o3(year)
    return da, date


def load_b2000_o3_climatology():
    path = B2000_ROOT / "partial_O3" / "B2000WCN_partial_O3_all_ranges.nc"
    if not path.exists():
        print(f"Missing B2000 O3 climatology source: {path}")
        return None
    with xr.open_dataset(path, decode_times=False) as ds:
        da = ds["O3_partial_60_90N_30_70hPa"]
        date = np.asarray(ds["date"].values, dtype=np.int64)
        _, mm, dd = date_parts(date)
        mmdd = np.array([m * 100 + d for m, d in zip(mm, dd)])
        da = da.assign_coords(mmdd=("time", mmdd))
        clim = da.groupby("mmdd").mean("time", skipna=True).load()
    all_mmdd = clim["mmdd"].values
    mask = (all_mmdd >= 101) & (all_mmdd <= 530)
    clim = clim.isel(mmdd=mask).rename({"mmdd": "lead_time"}).assign_coords(lead_time=np.arange(mask.sum()))
    return clim


def u60n10_from_file(path: Path, start_end=((1, 1), (5, 30))) -> Tuple[xr.DataArray, np.ndarray]:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask).sel(lat=60.0, method="nearest")
        date = date[mask]
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        u = interp_profile_logp(ds["U"].transpose("time", "lon", "lev"), p_mid.transpose("time", "lon", "lev"), PLEV_U_PA)
        u_zm = u.mean("lon", skipna=True).load()
    u_zm = u_zm.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return u_zm, date


def build_u60n10_case_cache(case, overwrite=False):
    out = CACHE_DIR / f"U60N10_{case}_members.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        date = np.asarray(da["date"].values, dtype=np.int64)
        return da, date
    files = sorted((HINDCAST_ROOT / case / "U").glob("*.U.nc"))
    if not files:
        raise FileNotFoundError(f"No U files for {case}")
    das, mids = [], []
    date0 = None
    for f in files:
        mid = member_short_id(f.name)
        print("U60N10", case, mid)
        da, date = u60n10_from_file(f)
        das.append(da)
        mids.append(mid)
        date0 = date
    out_da = xr.concat(das, dim=pd.Index(mids, name="member"))
    out_da.name = "U60N10"
    out_da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return out_da, date0


def build_u60n10_ref_cache(year=REF_YEAR, overwrite=False):
    out = CACHE_DIR / f"U60N10_BWCN_{year:04d}.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        return da, np.asarray(da["date"].values, dtype=np.int64)
    f = BWCN_ROOT / "U" / f"BWCN.cam.h3.{year:04d}.U.nc"
    da, date = u60n10_from_file(f)
    da.name = "U60N10"
    da.to_netcdf(out)
    return da, date


def plot_evolution(ref_data, ref_date, clim_data, experiments, ylabel, ylim, title, name, xlim=(0, 150)):
    fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
    xs = np.arange(len(ref_date))
    ax.plot(xs, ref_data.values, color="black", lw=3.5, label="Reference", zorder=10)
    if clim_data is not None:
        ax.plot(np.arange(clim_data.sizes[clim_data.dims[0]]), clim_data.values, color="black", ls=":", lw=3.2, label="Climatology", zorder=9)
    for exp in experiments:
        data, offset, color, label = exp["data"], exp["offset"], exp["color"], exp["label"]
        show_members = exp.get("show_members", True)
        show_spread = exp.get("show_spread", True)
        line_style = exp.get("ls", "-")
        if data is None:
            continue
        total = data.sizes["lead_time"]
        x = np.arange(offset, offset + total)
        keep = (x >= xlim[0]) & (x <= xlim[1])
        x = x[keep]
        sub = data.isel(lead_time=keep)
        if "member" in sub.dims:
            ens = sub.transpose("member", "lead_time")
            mean = ens.mean("member", skipna=True)
            std = ens.std("member", skipna=True)
            if show_members:
                for i in range(ens.sizes["member"]):
                    ax.plot(x, ens.isel(member=i).values, color=color, alpha=0.14, lw=1.0)
            if show_spread:
                ax.fill_between(x, (mean - std).values, (mean + std).values, color=color, alpha=0.20)
            ax.plot(x, mean.values, color=color, lw=3.2, ls=line_style, label=label)
        else:
            ax.plot(x, sub.values, color=color, lw=3.2, ls=line_style, label=label)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title, fontsize=15)
    ax.set_xticks([0, 31, 59, 90, 120, 151])
    ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
    ax.legend(fontsize=10)
    savefig(fig, name)
    plt.show()


def load_diag_for_case(diag, case):
    if diag == "O3":
        return load_case_o3_for_evolution(case)[0]
    if diag == "U60N10":
        da, _ = build_u60n10_case_cache(case)
        return da
    raise ValueError(diag)


def plot_jan_initial_best_worst(diag, data, ref, ref_date, clim, ylabel, ylim):
    best_da = subset_members_da(data, best5_members)
    worst_da = subset_members_da(data, worst5_members)
    experiments = [
        {"data": data, "label": "H-JanInit all 30 mean", "offset": 0, "color": "darkorange", "show_members": True, "show_spread": True},
        {"data": best_da, "label": "Best 5 mean", "offset": 0, "color": "dodgerblue", "show_members": False, "show_spread": False},
        {"data": worst_da, "label": "Worst 5 mean", "offset": 0, "color": "crimson", "show_members": False, "show_spread": False},
    ]
    plot_evolution(
        ref,
        ref_date,
        clim,
        experiments,
        ylabel,
        ylim,
        title=f"{diag}: JAN INITIAL ONLY all/best5/worst5 by O3 RMSE",
        name=f"{diag}_{CASE}_JAN_INITIAL_ONLY_all_best5_worst5_mean",
    )


evolution_specs = [
    {
        "name": "INT3D_vs_CLIM3D_0008_FebInit",
        "experiments": [
            ("0008-02", "H-INT-3D", 31, "forestgreen"),
            ("0008-02_NOCOUPL", "H-CLIM-3D", 31, "royalblue"),
        ],
    },
    {
        "name": "JAN_INITIAL_ONLY_0008",
        "experiments": [
            ("0008-01", "H-JanInit", 0, "darkorange"),
        ],
    },
]

for diag in ["O3", "U60N10"]:
    if diag == "O3":
        ref, ref_date = load_ref_o3_curve(REF_YEAR)
        clim = load_b2000_o3_climatology()
        ylabel, ylim = "Partial O3 column, 30-70 hPa (DU)", (60, 150)
    else:
        ref, ref_date = build_u60n10_ref_cache(REF_YEAR)
        old_clim_path = Path("/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache/U60N10/U60N10_B2000WCN_climatology.nc")
        clim = xr.open_dataarray(old_clim_path).load() if old_clim_path.exists() else None
        ylabel, ylim = "Zonal-mean U at 60N, 10 hPa (m s-1)", (-50, 90)
    for spec in evolution_specs:
        experiments = []
        jan_initial_data = None
        for case, label, offset, color in spec["experiments"]:
            try:
                data = load_diag_for_case(diag, case)
            except FileNotFoundError as exc:
                print(f"Skip {diag} {case}: {exc}")
                data = None
            if case == CASE:
                jan_initial_data = data
            experiments.append({"data": data, "label": label, "offset": offset, "color": color})
        plot_evolution(
            ref, ref_date, clim, experiments, ylabel, ylim,
            title=f"{diag}: {spec['name']}",
            name=f"{diag}_{spec['name']}",
        )
        if spec["name"] == "JAN_INITIAL_ONLY_0008" and jan_initial_data is not None:
            plot_jan_initial_best_worst(diag, jan_initial_data, ref, ref_date, clim, ylabel, ylim)
